### NuScene Data Preprocessing

In [2]:
import warnings
warnings.filterwarnings("ignore")

import os, sys
from pathlib import Path

repo_root = Path("/home/erdi/Dropbox/Docs/GitRepos/BEVFormer-Lab").resolve()
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from mmengine.config import Config
from mmengine.registry import init_default_scope

from tools.debug_train_bevformer_tiny_mini import (
    apply_config_fixes,
    register_runtime_compat_transforms,
    configure_train_dataset,
    build_dataset,
    attach_legacy_infos,
)

- Inspect the train dataset format. This is based on Nuscenes-mini

In [3]:
# insp
cfg = Config.fromfile("projects/configs/bevformer/bevformer_tiny.py")

apply_config_fixes(cfg)
init_default_scope("mmdet3d")
register_runtime_compat_transforms()

train_cfg = configure_train_dataset(cfg)
train_dataset = build_dataset(train_cfg)
metadata = attach_legacy_infos(train_dataset, train_cfg.ann_file, cfg.class_names)

print(type(train_dataset))
print(len(train_dataset))
print(metadata)

<class 'projects.mmdet3d_plugin.datasets.nuscenes_dataset.CustomNuScenesDataset'>
323
{'version': 'v1.0-mini'}


- Below we inspect a single sample.

In [5]:
# inspect a single `sample`
sample = train_dataset[0]

print(type(sample))
print(sample.keys())

<class 'dict'>
dict_keys(['img_metas', 'gt_bboxes_3d', 'gt_labels_3d', 'img'])


First start from those `img_metas`. Training sample is a temporal queue of 3 frames. Each frame has its own specs. Important ones are below

        filename          six camera image paths
        ori_shape         original image size
        img_shape         after resize
        pad_shape         after padding
        lidar2img         6 projection matrices, one per camera
        can_bus           (controller area network) ego-motion/state info(position,orientation,velocity etc)
                           for temporal BEV alignment
        prev_bev_exists   whether this frame can use previous BEV
        sample_idx        nuScenes sample token
        scene_token       scene id

In [6]:
img_meta = sample["img_metas"].data
print("temporal queue of three frames:", img_meta.keys())

# the first sample in this queue, (t-1)
print("Each sample has the following keys:\n")
m = sample["img_metas"].data[0]
print(list(m.keys()))

print('\n------KEY INFO---------\n')
print("sample_idx:", m["sample_idx"])
print("scene_token:", m["scene_token"])
print("prev_bev_exists:", m["prev_bev_exists"])
print("num cameras:", len(m["filename"]))
print("ori_shape:", m["ori_shape"][0])
print("img_shape:", m["img_shape"][0])
print("pad_shape:", m["pad_shape"][0])
print("can_bus shape:", m["can_bus"].shape)
print("lidar2img len:", len(m["lidar2img"]))
print("lidar2img[0] shape:", m["lidar2img"][0].shape)

temporal queue of three frames: dict_keys([0, 1, 2])
Each sample has the following keys:

['filename', 'ori_shape', 'img_shape', 'lidar2img', 'lidar2cam', 'pad_shape', 'box_mode_3d', 'box_type_3d', 'img_norm_cfg', 'sample_idx', 'prev_idx', 'next_idx', 'pts_filename', 'scene_token', 'can_bus', 'prev_bev_exists']

------KEY INFO---------

sample_idx: ca9a282c9e77460f8360f564131a8af5
scene_token: cc8c0bf57f984915a77078b10eb33198
prev_bev_exists: False
num cameras: 6
ori_shape: (450, 800, 3)
img_shape: (480, 800, 3)
pad_shape: (480, 800, 3)
can_bus shape: (18,)
lidar2img len: 6
lidar2img[0] shape: (4, 4)


In [7]:
m = sample["img_metas"].data[0]
print("sample_idx:", m["sample_idx"])
print("scene_token:", m["scene_token"])
print("prev_bev_exists:", m["prev_bev_exists"])
print("num cameras:", len(m["filename"]))
print("ori_shape:", m["ori_shape"][0])
print("img_shape:", m["img_shape"][0])
print("pad_shape:", m["pad_shape"][0])
print("can_bus shape:", m["can_bus"].shape)
print("lidar2img len:", len(m["lidar2img"]))
print("lidar2img[0] shape:", m["lidar2img"][0].shape)

sample_idx: ca9a282c9e77460f8360f564131a8af5
scene_token: cc8c0bf57f984915a77078b10eb33198
prev_bev_exists: False
num cameras: 6
ori_shape: (450, 800, 3)
img_shape: (480, 800, 3)
pad_shape: (480, 800, 3)
can_bus shape: (18,)
lidar2img len: 6
lidar2img[0] shape: (4, 4)


- We can now inspect other keys

        temporal multi-camera image tensor -->  (3, 6, 3, 480, 800)
        3D ground-truth boxes -->               (N, 9) with (x, y, z, dx, dy, dz, yaw, vx, vy)
        Class labels -->                        (N,) with (1,3,4,...)

In [8]:
img = sample["img"].data
boxes = sample["gt_bboxes_3d"].data
labels = sample["gt_labels_3d"].data

print("img:", type(img), img.shape)


print("boxes tensor:", boxes.tensor.shape)
print("label tensor:", labels.shape)


img: <class 'torch.Tensor'> torch.Size([3, 6, 3, 480, 800])
boxes tensor: torch.Size([50, 9])
label tensor: torch.Size([50])


### Data Augmentation

    LoadMultiViewImageFromFilesCompat   : loads the 6 synchronized camera images from disk

    PhotoMetricDistortionMultiViewImage : random brightness/contrast/saturation/hue augmentation applied consistently
                                          to all camera views

    LoadAnnotations3D                   : loads 3D ground-truth boxes and class labels (gt_bboxes_3d, gt_labels_3d)

    ObjectRangeFilter                   : removes objects outside the predefined BEV region
                                          (≈102m × 102m around the ego vehicle)

    ObjectNameFilter                    : keeps only the 10 nuScenes detection classes

    NormalizeMultiviewImage             : ImageNet-style normalization applied to all camera images

    RandomScaleImageMultiViewImage      : resizes all camera images (0.5×) and updates projection geometry (e.g. lidar2img)

    PadMultiViewImage                   : pads images so height/width are divisible by 32

    DefaultFormatBundle3DCompat         : converts 6 × H × W × C images → 6 × C × H × W PyTorch tensor; wraps labels/boxes

    CustomCollect3D                     : collects the final model inputs (img, gt_bboxes_3d, gt_labels_3d, img_metas)

In [9]:
for i, t in enumerate(train_dataset.pipeline.transforms):
    print(i,t)
    print()

0 <tools.debug_train_bevformer_tiny_mini.register_runtime_compat_transforms.<locals>.LoadMultiViewImageFromFilesCompat object at 0x7556e15dde40>

1 PhotoMetricDistortionMultiViewImage(
brightness_delta=32,
contrast_range=(0.5, 1.5),
saturation_range=(0.5, 1.5),
hue_delta=18)

2 LoadAnnotations3D(
    with_bbox_3d=True,     with_label_3d=True,     with_attr_label=False,     with_mask_3d=False,     with_seg_3d=False,     with_panoptic_3d=False,     with_bbox=False,     with_label=False,     with_mask=False,     with_seg=False,     with_bbox_depth=False,     poly2mask=True)    seg_offset=None)

3 ObjectRangeFilter(point_cloud_range=[-51.20000076293945, -51.20000076293945, -5.0, 51.20000076293945, 51.20000076293945, 3.0])

4 ObjectNameFilter(classes=['car', 'truck', 'construction_vehicle', 'bus', 'trailer', 'barrier', 'motorcycle', 'bicycle', 'pedestrian', 'traffic_cone'])

5 NormalizeMultiviewImage(mean=[123.675 116.28  103.53 ], std=[58.395 57.12  57.375], to_rgb=True)

6 RandomScaleImag

## Data Flow Through the Model